In [ ]:
""" 筛选杨浦区内的百度街景图片，并复制对应图片到新文件夹。

输入：
1. 街景 CSV：
   D:/街景/3.shanghai_only__2019.csv

2. 原始街景图片文件夹：
   D:/街景/2019

3. 杨浦区行政边界文件：
   D:/街景/yangpu_boundary.geojson

输出：
1. 杨浦区街景点 CSV：
   D:/街景/yangpu_streetview_2019.csv

2. 杨浦区街景图片文件夹：
   D:/街景/yangpu_2019
"""

from __future__ import annotations

import math
import shutil
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from tqdm import tqdm


# =========================
# 1. 路径配置
# =========================

CSV_PATH = Path(r"D:\街景\3.shanghai_only__2019.csv")
IMAGE_DIR = Path(r"D:\街景\2019")

# 你需要提前准备这个杨浦区边界文件
YANGPU_BOUNDARY_PATH = Path(r"D:\街景\杨浦区_县.geojson")

OUTPUT_CSV_PATH = Path(r"D:\街景\yangpu_streetview_2019.csv")
OUTPUT_IMAGE_DIR = Path(r"D:\街景\yangpu_2019")

# 你的街景 CSV 坐标系是百度 BD-09
POINT_COORD_SYSTEM = "BD09"

# 边界文件坐标系：
# 如果你的 yangpu_boundary.geojson 是常见 WGS84，经纬度坐标，保持 WGS84。
# 如果边界文件来自百度地图并且也是 BD-09，可以改成 BD09。
# 如果边界文件来自高德/腾讯，可能是 GCJ02，可以改成 GCJ02。
BOUNDARY_COORD_SYSTEM = "WGS84"

# 支持的图片后缀
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".webp"]


# =========================
# 2. 坐标转换函数
# =========================

X_PI = math.pi * 3000.0 / 180.0
PI = math.pi
A = 6378245.0
EE = 0.00669342162296594323


def out_of_china(lon: float, lat: float) -> bool:
    """判断是否在中国范围外。"""
    return not (73.66 < lon < 135.05 and 3.86 < lat < 53.55)


def transform_lat(lon: float, lat: float) -> float:
    ret = (
        -100.0
        + 2.0 * lon
        + 3.0 * lat
        + 0.2 * lat * lat
        + 0.1 * lon * lat
        + 0.2 * math.sqrt(abs(lon))
    )
    ret += (
        (20.0 * math.sin(6.0 * lon * PI) + 20.0 * math.sin(2.0 * lon * PI))
        * 2.0
        / 3.0
    )
    ret += (
        (20.0 * math.sin(lat * PI) + 40.0 * math.sin(lat / 3.0 * PI))
        * 2.0
        / 3.0
    )
    ret += (
        (160.0 * math.sin(lat / 12.0 * PI) + 320.0 * math.sin(lat * PI / 30.0))
        * 2.0
        / 3.0
    )
    return ret


def transform_lon(lon: float, lat: float) -> float:
    ret = (
        300.0
        + lon
        + 2.0 * lat
        + 0.1 * lon * lon
        + 0.1 * lon * lat
        + 0.1 * math.sqrt(abs(lon))
    )
    ret += (
        (20.0 * math.sin(6.0 * lon * PI) + 20.0 * math.sin(2.0 * lon * PI))
        * 2.0
        / 3.0
    )
    ret += (
        (20.0 * math.sin(lon * PI) + 40.0 * math.sin(lon / 3.0 * PI))
        * 2.0
        / 3.0
    )
    ret += (
        (150.0 * math.sin(lon / 12.0 * PI) + 300.0 * math.sin(lon / 30.0 * PI))
        * 2.0
        / 3.0
    )
    return ret


def bd09_to_gcj02(bd_lon: float, bd_lat: float) -> tuple[float, float]:
    """百度 BD-09 转 GCJ-02。"""
    x = bd_lon - 0.0065
    y = bd_lat - 0.006
    z = math.sqrt(x * x + y * y) - 0.00002 * math.sin(y * X_PI)
    theta = math.atan2(y, x) - 0.000003 * math.cos(x * X_PI)
    gcj_lon = z * math.cos(theta)
    gcj_lat = z * math.sin(theta)
    return gcj_lon, gcj_lat


def gcj02_to_wgs84(gcj_lon: float, gcj_lat: float) -> tuple[float, float]:
    """GCJ-02 转 WGS84。"""
    if out_of_china(gcj_lon, gcj_lat):
        return gcj_lon, gcj_lat

    dlat = transform_lat(gcj_lon - 105.0, gcj_lat - 35.0)
    dlon = transform_lon(gcj_lon - 105.0, gcj_lat - 35.0)
    radlat = gcj_lat / 180.0 * PI
    magic = math.sin(radlat)
    magic = 1 - EE * magic * magic
    sqrt_magic = math.sqrt(magic)

    dlat = (dlat * 180.0) / ((A * (1 - EE)) / (magic * sqrt_magic) * PI)
    dlon = (dlon * 180.0) / (A / sqrt_magic * math.cos(radlat) * PI)

    mg_lat = gcj_lat + dlat
    mg_lon = gcj_lon + dlon

    wgs_lon = gcj_lon * 2 - mg_lon
    wgs_lat = gcj_lat * 2 - mg_lat
    return wgs_lon, wgs_lat


def bd09_to_wgs84(bd_lon: float, bd_lat: float) -> tuple[float, float]:
    """百度 BD-09 转 WGS84。"""
    gcj_lon, gcj_lat = bd09_to_gcj02(bd_lon, bd_lat)
    return gcj02_to_wgs84(gcj_lon, gcj_lat)


def convert_point_coord(
    lon: float,
    lat: float,
    from_system: str,
    to_system: str,
) -> tuple[float, float]:
    """
    将点坐标转换到目标坐标系。

    当前支持：
    - BD09 -> WGS84
    - BD09 -> GCJ02
    - BD09 -> BD09
    - WGS84 -> WGS84
    """
    from_system = from_system.upper()
    to_system = to_system.upper()

    if from_system == to_system:
        return lon, lat

    if from_system == "BD09" and to_system == "WGS84":
        return bd09_to_wgs84(lon, lat)

    if from_system == "BD09" and to_system == "GCJ02":
        return bd09_to_gcj02(lon, lat)

    raise ValueError(f"暂不支持从 {from_system} 转换到 {to_system}")


# =========================
# 3. 工具函数
# =========================

def normalize_fid(fid) -> str:
    """
    将 FID 规范化为图片文件名 stem。

    例如：
    1 -> "1"
    1.0 -> "1"
    "0001" -> "0001"
    """
    if pd.isna(fid):
        return ""

    fid_str = str(fid).strip()

    # 处理 Excel / CSV 可能把整数读成 1.0 的情况
    try:
        float_value = float(fid_str)
        int_value = int(float_value)
        if float_value == int_value:
            return str(int_value)
    except ValueError:
        pass

    return fid_str


def build_image_index(image_dir: Path) -> dict[str, Path]:
    """
    建立图片索引：
    key: 文件名不含后缀，例如 "1"
    value: 图片完整路径
    """
    if not image_dir.exists():
        raise FileNotFoundError(f"图片文件夹不存在：{image_dir}")

    image_index: dict[str, Path] = {}

    for path in image_dir.iterdir():
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            image_index[path.stem] = path

    return image_index


# =========================
# 4. 主流程
# =========================

def main() -> None:
    print("开始读取街景 CSV...")

    if not CSV_PATH.exists():
        raise FileNotFoundError(f"CSV 文件不存在：{CSV_PATH}")

    if not YANGPU_BOUNDARY_PATH.exists():
        raise FileNotFoundError(
            f"杨浦区边界文件不存在：{YANGPU_BOUNDARY_PATH}\n"
            "请先准备杨浦区行政边界 GeoJSON 或 Shapefile。"
        )

    # sep=None 可以自动识别逗号、制表符等分隔符
    df = pd.read_csv(CSV_PATH, sep=None, engine="python", encoding="utf-8")

    # 去掉字段名两端空格
    df.columns = [c.strip() for c in df.columns]

    required_cols = ["FID", "lon", "lat", "year", "month", "svid", "north_angle"]
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"CSV 缺少必要字段：{missing_cols}")

    print(f"原始记录数：{len(df)}")

    # 转换 lon / lat 为数值
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")

    # 删除无效坐标
    df = df.dropna(subset=["lon", "lat"]).copy()
    print(f"删除无效坐标后记录数：{len(df)}")

    print("开始坐标转换...")

    if BOUNDARY_COORD_SYSTEM.upper() == "WGS84":
        converted = df.apply(
            lambda row: convert_point_coord(
                row["lon"],
                row["lat"],
                from_system=POINT_COORD_SYSTEM,
                to_system="WGS84",
            ),
            axis=1,
        )
        df["lon_wgs84"] = [item[0] for item in converted]
        df["lat_wgs84"] = [item[1] for item in converted]

        join_lon_col = "lon_wgs84"
        join_lat_col = "lat_wgs84"

    elif BOUNDARY_COORD_SYSTEM.upper() == "GCJ02":
        converted = df.apply(
            lambda row: convert_point_coord(
                row["lon"],
                row["lat"],
                from_system=POINT_COORD_SYSTEM,
                to_system="GCJ02",
            ),
            axis=1,
        )
        df["lon_gcj02"] = [item[0] for item in converted]
        df["lat_gcj02"] = [item[1] for item in converted]

        join_lon_col = "lon_gcj02"
        join_lat_col = "lat_gcj02"

    elif BOUNDARY_COORD_SYSTEM.upper() == "BD09":
        join_lon_col = "lon"
        join_lat_col = "lat"

    else:
        raise ValueError(
            "BOUNDARY_COORD_SYSTEM 只能设置为 WGS84、GCJ02 或 BD09"
        )

    print("开始读取杨浦区边界...")

    boundary_gdf = gpd.read_file(YANGPU_BOUNDARY_PATH)

    if boundary_gdf.empty:
        raise ValueError("杨浦区边界文件为空。")

    # 如果边界文件带有 CRS，统一转成 EPSG:4326。
    # 注意：这里的 EPSG:4326 只是经纬度坐标框架。
    # 若边界坐标实际是 GCJ02 或 BD09，需要通过 BOUNDARY_COORD_SYSTEM 保证点坐标一致。
    if boundary_gdf.crs is None:
        boundary_gdf = boundary_gdf.set_crs(epsg=4326)
    else:
        boundary_gdf = boundary_gdf.to_crs(epsg=4326)

    boundary_gdf = boundary_gdf[["geometry"]].copy()
    boundary_gdf["boundary_id"] = 1

    print("开始构建街景点 GeoDataFrame...")

    geometry = [
        Point(lon, lat)
        for lon, lat in zip(df[join_lon_col], df[join_lat_col])
    ]

    point_gdf = gpd.GeoDataFrame(
        df,
        geometry=geometry,
        crs="EPSG:4326",
    )

    print("开始空间筛选杨浦区内街景点...")

    # 对点而言 intersects 可以包含边界线上的点
    yangpu_points = gpd.sjoin(
        point_gdf,
        boundary_gdf,
        how="inner",
        predicate="intersects",
    )

    # 清理空间连接产生的字段
    drop_cols = [c for c in ["index_right", "boundary_id"] if c in yangpu_points.columns]
    yangpu_points = yangpu_points.drop(columns=drop_cols)

    print(f"杨浦区内街景点数量：{len(yangpu_points)}")

    # 输出 CSV 前，生成标准 FID 字符串
    yangpu_points["FID_str"] = yangpu_points["FID"].apply(normalize_fid)

    # 输出 CSV 不需要 geometry 对象，保留坐标字段即可
    output_df = pd.DataFrame(yangpu_points.drop(columns="geometry"))

    # 为后续 Agent 使用，增加图片文件名字段
    output_df["image_file_stem"] = output_df["FID_str"]

    print(f"保存杨浦区街景 CSV：{OUTPUT_CSV_PATH}")
    OUTPUT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    output_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")

    print("开始建立图片索引...")
    image_index = build_image_index(IMAGE_DIR)

    print(f"原始图片数量：{len(image_index)}")

    OUTPUT_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

    copied_count = 0
    missing_count = 0
    missing_fids: list[str] = []

    print("开始复制杨浦区街景图片...")

    for fid in tqdm(output_df["FID_str"].tolist()):
        if fid in image_index:
            src_path = image_index[fid]
            dst_path = OUTPUT_IMAGE_DIR / src_path.name
            shutil.copy2(src_path, dst_path)
            copied_count += 1
        else:
            missing_count += 1
            missing_fids.append(fid)

    print("图片复制完成。")
    print(f"成功复制图片数量：{copied_count}")
    print(f"未找到图片数量：{missing_count}")

    if missing_fids:
        missing_path = OUTPUT_CSV_PATH.parent / "yangpu_missing_image_fids.txt"
        with open(missing_path, "w", encoding="utf-8") as f:
            for fid in missing_fids:
                f.write(f"{fid}\n")

        print(f"未找到图片的 FID 已保存：{missing_path}")

    print("全部处理完成。")


if __name__ == "__main__":
    main()

In [ ]:
"""02_prepare_streetview_metadata.py

功能：
1. 读取已经筛选出的杨浦区街景 CSV；
2. 检查对应图片是否存在；
3. 检查图片是否可以正常打开；
4. 标准化 image_id、file_path、坐标、时间、角度等字段；
5. 输出 image_metadata.csv / parquet / geojson；
6. 生成 image_analysis_template.csv；
7. 生成 multimodal_jobs.jsonl，供后续多模态模型分析使用；
8. 输出数据质量报告。

输入：
- D:/街景/yangpu_streetview_2019.csv
- D:/街景/yangpu_2019/

输出：
- D:/街景/yangpu_processed_2019/
"""

from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any

import geopandas as gpd
import pandas as pd
from PIL import Image
from shapely.geometry import Point
from tqdm import tqdm


# =========================
# 1. 路径配置
# =========================

INPUT_CSV = Path(r"D:\街景\yangpu_streetview_2019.csv")
IMAGE_DIR = Path(r"D:\街景\yangpu_2019")
OUTPUT_DIR = Path(r"D:\街景\yangpu_processed_2019")

OUTPUT_METADATA_CSV = OUTPUT_DIR / "image_metadata.csv"
OUTPUT_METADATA_PARQUET = OUTPUT_DIR / "image_metadata.parquet"
OUTPUT_METADATA_GEOJSON = OUTPUT_DIR / "image_metadata.geojson"

OUTPUT_ANALYSIS_TEMPLATE_CSV = OUTPUT_DIR / "image_analysis_template.csv"
OUTPUT_ANALYSIS_TEMPLATE_JSONL = OUTPUT_DIR / "image_analysis_template.jsonl"
OUTPUT_MULTIMODAL_JOBS_JSONL = OUTPUT_DIR / "multimodal_jobs.jsonl"
OUTPUT_QUALITY_REPORT = OUTPUT_DIR / "streetview_quality_report.md"

IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".webp"]

# 如果你的筛选结果中已经有 lon_wgs84 / lat_wgs84，则优先使用它们。
# 如果没有，则会把原始 lon / lat 当作 BD-09，并转换为 WGS84。
RAW_COORD_SYSTEM = "BD09"


# =========================
# 2. 百度 BD-09 坐标转换函数
# =========================

X_PI = math.pi * 3000.0 / 180.0
PI = math.pi
A = 6378245.0
EE = 0.00669342162296594323


def out_of_china(lon: float, lat: float) -> bool:
    return not (73.66 < lon < 135.05 and 3.86 < lat < 53.55)


def transform_lat(lon: float, lat: float) -> float:
    ret = (
        -100.0
        + 2.0 * lon
        + 3.0 * lat
        + 0.2 * lat * lat
        + 0.1 * lon * lat
        + 0.2 * math.sqrt(abs(lon))
    )
    ret += (
        (20.0 * math.sin(6.0 * lon * PI) + 20.0 * math.sin(2.0 * lon * PI))
        * 2.0
        / 3.0
    )
    ret += (
        (20.0 * math.sin(lat * PI) + 40.0 * math.sin(lat / 3.0 * PI))
        * 2.0
        / 3.0
    )
    ret += (
        (160.0 * math.sin(lat / 12.0 * PI) + 320.0 * math.sin(lat * PI / 30.0))
        * 2.0
        / 3.0
    )
    return ret


def transform_lon(lon: float, lat: float) -> float:
    ret = (
        300.0
        + lon
        + 2.0 * lat
        + 0.1 * lon * lon
        + 0.1 * lon * lat
        + 0.1 * math.sqrt(abs(lon))
    )
    ret += (
        (20.0 * math.sin(6.0 * lon * PI) + 20.0 * math.sin(2.0 * lon * PI))
        * 2.0
        / 3.0
    )
    ret += (
        (20.0 * math.sin(lon * PI) + 40.0 * math.sin(lon / 3.0 * PI))
        * 2.0
        / 3.0
    )
    ret += (
        (150.0 * math.sin(lon / 12.0 * PI) + 300.0 * math.sin(lon / 30.0 * PI))
        * 2.0
        / 3.0
    )
    return ret


def bd09_to_gcj02(bd_lon: float, bd_lat: float) -> tuple[float, float]:
    x = bd_lon - 0.0065
    y = bd_lat - 0.006
    z = math.sqrt(x * x + y * y) - 0.00002 * math.sin(y * X_PI)
    theta = math.atan2(y, x) - 0.000003 * math.cos(x * X_PI)
    gcj_lon = z * math.cos(theta)
    gcj_lat = z * math.sin(theta)
    return gcj_lon, gcj_lat


def gcj02_to_wgs84(gcj_lon: float, gcj_lat: float) -> tuple[float, float]:
    if out_of_china(gcj_lon, gcj_lat):
        return gcj_lon, gcj_lat

    dlat = transform_lat(gcj_lon - 105.0, gcj_lat - 35.0)
    dlon = transform_lon(gcj_lon - 105.0, gcj_lat - 35.0)

    radlat = gcj_lat / 180.0 * PI
    magic = math.sin(radlat)
    magic = 1 - EE * magic * magic
    sqrt_magic = math.sqrt(magic)

    dlat = (dlat * 180.0) / ((A * (1 - EE)) / (magic * sqrt_magic) * PI)
    dlon = (dlon * 180.0) / (A / sqrt_magic * math.cos(radlat) * PI)

    mg_lat = gcj_lat + dlat
    mg_lon = gcj_lon + dlon

    wgs_lon = gcj_lon * 2 - mg_lon
    wgs_lat = gcj_lat * 2 - mg_lat
    return wgs_lon, wgs_lat


def bd09_to_wgs84(bd_lon: float, bd_lat: float) -> tuple[float, float]:
    gcj_lon, gcj_lat = bd09_to_gcj02(bd_lon, bd_lat)
    return gcj02_to_wgs84(gcj_lon, gcj_lat)


# =========================
# 3. 基础工具函数
# =========================

def normalize_fid(fid: Any) -> str:
    """
    规范化 FID。
    例如：
    1 -> "1"
    1.0 -> "1"
    """
    if pd.isna(fid):
        return ""

    fid_str = str(fid).strip()

    try:
        float_value = float(fid_str)
        int_value = int(float_value)
        if float_value == int_value:
            return str(int_value)
    except ValueError:
        pass

    return fid_str


def build_image_index(image_dir: Path) -> dict[str, Path]:
    """
    根据图片文件名建立索引。
    key: 文件名不含后缀，例如 "1"
    value: 图片完整路径
    """
    if not image_dir.exists():
        raise FileNotFoundError(f"图片文件夹不存在：{image_dir}")

    image_index: dict[str, Path] = {}

    for path in image_dir.iterdir():
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            image_index[path.stem] = path

    return image_index


def angle_to_direction(angle: Any) -> str:
    """
    将角度转换为方向标签。
    0/360 = 北，90 = 东，180 = 南，270 = 西。
    """
    if pd.isna(angle):
        return "unknown"

    try:
        a = float(angle) % 360
    except ValueError:
        return "unknown"

    directions = [
        ("N", 337.5, 360.0),
        ("N", 0.0, 22.5),
        ("NE", 22.5, 67.5),
        ("E", 67.5, 112.5),
        ("SE", 112.5, 157.5),
        ("S", 157.5, 202.5),
        ("SW", 202.5, 247.5),
        ("W", 247.5, 292.5),
        ("NW", 292.5, 337.5),
    ]

    for label, start, end in directions:
        if start <= a < end:
            return label

    return "unknown"


def check_image(path: Path) -> dict[str, Any]:
    """
    检查图片是否可正常打开，并读取宽高。
    """
    result = {
        "image_exists": False,
        "image_readable": False,
        "image_width": None,
        "image_height": None,
        "image_error": "",
    }

    if not path.exists():
        result["image_error"] = "file_not_found"
        return result

    result["image_exists"] = True

    try:
        with Image.open(path) as img:
            result["image_width"] = img.width
            result["image_height"] = img.height
            img.verify()
        result["image_readable"] = True
    except Exception as exc:
        result["image_error"] = str(exc)

    return result


def make_multimodal_prompt() -> str:
    """
    给后续多模态模型使用的统一提示词。
    这里只生成任务，不直接调用模型。
    """
    return (
        "请分析这张上海市杨浦区街景图片，从城市更新、老年友好、步行友好和15分钟生活圈角度，"
        "识别图片中的空间环境问题。请特别关注："
        "1. 人行道是否连续、宽度是否充足；"
        "2. 是否存在非机动车或杂物占用人行空间；"
        "3. 是否有无障碍坡道、盲道、扶手等设施；"
        "4. 是否有座椅、遮阴、绿化等适老化设施；"
        "5. 是否存在路面破损、过街不安全、标识不清等问题；"
        "6. 街道界面是否有活力。"
        "请输出结构化 JSON，不要输出多余解释。"
    )


# =========================
# 4. 主处理流程
# =========================

def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    if not INPUT_CSV.exists():
        raise FileNotFoundError(f"输入 CSV 不存在：{INPUT_CSV}")

    print("读取杨浦区街景 CSV...")
    df = pd.read_csv(INPUT_CSV, sep=None, engine="python", encoding="utf-8")
    df.columns = [c.strip() for c in df.columns]

    required_cols = ["FID", "lon", "lat", "year", "month", "svid", "north_angle"]
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"输入 CSV 缺少必要字段：{missing_cols}")

    original_count = len(df)

    print("规范化字段...")
    df["FID_str"] = df["FID"].apply(normalize_fid)
    df["image_id"] = df["FID_str"].apply(lambda x: f"sv_2019_{x}")

    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df["month"] = pd.to_numeric(df["month"], errors="coerce").astype("Int64")
    df["north_angle"] = pd.to_numeric(df["north_angle"], errors="coerce")

    invalid_coord_count = df[["lon", "lat"]].isna().any(axis=1).sum()
    df = df.dropna(subset=["lon", "lat"]).copy()

    print("处理 WGS84 坐标...")

    if "lon_wgs84" in df.columns and "lat_wgs84" in df.columns:
        df["lon_wgs84"] = pd.to_numeric(df["lon_wgs84"], errors="coerce")
        df["lat_wgs84"] = pd.to_numeric(df["lat_wgs84"], errors="coerce")

        missing_wgs = df[["lon_wgs84", "lat_wgs84"]].isna().any(axis=1)

        if missing_wgs.any():
            converted = df.loc[missing_wgs].apply(
                lambda row: bd09_to_wgs84(row["lon"], row["lat"]),
                axis=1,
            )
            df.loc[missing_wgs, "lon_wgs84"] = [x[0] for x in converted]
            df.loc[missing_wgs, "lat_wgs84"] = [x[1] for x in converted]

    else:
        converted = df.apply(
            lambda row: bd09_to_wgs84(row["lon"], row["lat"]),
            axis=1,
        )
        df["lon_wgs84"] = [x[0] for x in converted]
        df["lat_wgs84"] = [x[1] for x in converted]

    print("建立图片索引...")
    image_index = build_image_index(IMAGE_DIR)

    print("匹配图片路径...")
    df["image_file_name"] = df["FID_str"].apply(
        lambda fid: image_index[fid].name if fid in image_index else ""
    )
    df["image_abs_path"] = df["FID_str"].apply(
        lambda fid: str(image_index[fid]) if fid in image_index else ""
    )
    df["image_rel_path"] = df["image_file_name"].apply(
        lambda name: str(Path("images") / name) if name else ""
    )

    print("检查图片可读性...")
    image_check_results = []

    for path_str in tqdm(df["image_abs_path"].tolist()):
        if path_str:
            image_check_results.append(check_image(Path(path_str)))
        else:
            image_check_results.append(
                {
                    "image_exists": False,
                    "image_readable": False,
                    "image_width": None,
                    "image_height": None,
                    "image_error": "missing_image_path",
                }
            )

    image_check_df = pd.DataFrame(image_check_results)
    df = pd.concat([df.reset_index(drop=True), image_check_df], axis=1)

    print("生成时间和方向字段...")
    df["capture_year"] = df["year"]
    df["capture_month"] = df["month"]
    df["capture_ym"] = df.apply(
        lambda row: (
            f"{int(row['capture_year']):04d}-{int(row['capture_month']):02d}"
            if not pd.isna(row["capture_year"]) and not pd.isna(row["capture_month"])
            else ""
        ),
        axis=1,
    )
    df["direction_bucket"] = df["north_angle"].apply(angle_to_direction)

    print("添加项目字段...")
    df["district"] = "杨浦区"
    df["city"] = "上海市"
    df["source"] = "baidu_streetview_api"
    df["source_crs"] = RAW_COORD_SYSTEM
    df["analysis_crs"] = "WGS84"
    df["data_year"] = 2019

    print("构建 GeoDataFrame...")
    geometry = [
        Point(lon, lat)
        for lon, lat in zip(df["lon_wgs84"], df["lat_wgs84"])
    ]

    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

    # 按 image_id 去重
    duplicate_image_id_count = gdf["image_id"].duplicated().sum()
    if duplicate_image_id_count > 0:
        gdf = gdf.drop_duplicates(subset=["image_id"], keep="first").copy()

    # 推荐 metadata 字段顺序
    metadata_cols = [
        "image_id",
        "FID",
        "FID_str",
        "svid",
        "image_file_name",
        "image_abs_path",
        "image_rel_path",
        "lon",
        "lat",
        "lon_wgs84",
        "lat_wgs84",
        "year",
        "month",
        "capture_year",
        "capture_month",
        "capture_ym",
        "north_angle",
        "direction_bucket",
        "city",
        "district",
        "source",
        "source_crs",
        "analysis_crs",
        "data_year",
        "image_exists",
        "image_readable",
        "image_width",
        "image_height",
        "image_error",
        "geometry",
    ]

    existing_cols = [c for c in metadata_cols if c in gdf.columns]
    gdf = gdf[existing_cols].copy()

    print("保存 image_metadata.csv...")
    csv_df = pd.DataFrame(gdf.drop(columns="geometry"))
    csv_df.to_csv(OUTPUT_METADATA_CSV, index=False, encoding="utf-8-sig")

    print("保存 image_metadata.parquet...")
    gdf.to_parquet(OUTPUT_METADATA_PARQUET, index=False)

    print("保存 image_metadata.geojson...")
    gdf.to_file(OUTPUT_METADATA_GEOJSON, driver="GeoJSON", encoding="utf-8")

    print("生成 image_analysis_template...")

    analysis_template = pd.DataFrame(
        {
            "image_id": gdf["image_id"],
            "FID_str": gdf["FID_str"],
            "image_abs_path": gdf["image_abs_path"],
            "caption": "",
            "detected_issues": "",
            "sidewalk_continuity": "unknown",
            "barrier_free": "unknown",
            "bike_conflict": "unknown",
            "shade": "unknown",
            "greenery": "unknown",
            "lighting": "unknown",
            "crossing_safety": "unknown",
            "seating": "unknown",
            "road_surface": "unknown",
            "elderly_friendly_score": "",
            "walkability_score": "",
            "analysis_status": "not_analyzed",
            "analysis_model": "",
            "analysis_time": "",
        }
    )

    analysis_template.to_csv(
        OUTPUT_ANALYSIS_TEMPLATE_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    with open(OUTPUT_ANALYSIS_TEMPLATE_JSONL, "w", encoding="utf-8") as f:
        for _, row in analysis_template.iterrows():
            record = row.to_dict()
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print("生成 multimodal_jobs.jsonl...")

    prompt = make_multimodal_prompt()

    with open(OUTPUT_MULTIMODAL_JOBS_JSONL, "w", encoding="utf-8") as f:
        for _, row in gdf.iterrows():
            if not row["image_readable"]:
                continue

            job = {
                "image_id": row["image_id"],
                "FID_str": row["FID_str"],
                "image_abs_path": row["image_abs_path"],
                "lon_wgs84": row["lon_wgs84"],
                "lat_wgs84": row["lat_wgs84"],
                "capture_ym": row["capture_ym"],
                "north_angle": row["north_angle"],
                "direction_bucket": row["direction_bucket"],
                "prompt": prompt,
            }
            f.write(json.dumps(job, ensure_ascii=False) + "\n")

    print("生成质量报告...")

    final_count = len(gdf)
    image_exists_count = int(gdf["image_exists"].sum())
    image_readable_count = int(gdf["image_readable"].sum())
    missing_image_count = final_count - image_exists_count
    unreadable_image_count = image_exists_count - image_readable_count

    direction_stats = gdf["direction_bucket"].value_counts(dropna=False).to_dict()
    month_stats = gdf["capture_ym"].value_counts(dropna=False).sort_index().to_dict()

    report = f"""# 杨浦区街景数据预处理质量报告

## 1. 输入数据

- 输入 CSV：`{INPUT_CSV}`
- 输入图片文件夹：`{IMAGE_DIR}`
- 输出目录：`{OUTPUT_DIR}`

## 2. 数据数量

| 指标 | 数量 |
|---|---:|
| 原始 CSV 记录数 | {original_count} |
| 无效坐标记录数 | {invalid_coord_count} |
| 去重后 metadata 记录数 | {final_count} |
| 图片存在数量 | {image_exists_count} |
| 图片可正常读取数量 | {image_readable_count} |
| 缺失图片数量 | {missing_image_count} |
| 不可读取图片数量 | {unreadable_image_count} |
| 重复 image_id 数量 | {duplicate_image_id_count} |

## 3. 坐标说明

- 原始坐标字段：`lon`, `lat`
- 原始坐标系：`{RAW_COORD_SYSTEM}`
- 分析坐标字段：`lon_wgs84`, `lat_wgs84`
- 分析坐标系：`WGS84`

## 4. 方向分布

```json
{json.dumps(direction_stats, ensure_ascii=False, indent=2)}
{json.dumps(month_stats, ensure_ascii=False, indent=2)}
"""

    OUTPUT_QUALITY_REPORT.write_text(report, encoding="utf-8")
    print(f"✅ 质量报告已保存：{OUTPUT_QUALITY_REPORT}")

if __name__ == "__main__":
    main()